# SalesInsight PY

*Análise e visualização de dados de vendas*

**Autor:** Bruno Miguel Corrêa

**Curso:** Desenvolvimento de IA para Análise Preditiva — SENAI/SC


## Visão Geral

O SalesInsight PY estrutura um fluxo completo de análise de dados de vendas, desde a inspeção da base bruta até a geração de métricas, segmentações e visualizações.

O processamento utiliza Python, Pandas e NumPy. Os resultados gráficos foram produzidos com Matplotlib e Seaborn.


## 1. Configuração do Ambiente

Importação das dependências e identificação das versões utilizadas na execução.


In [50]:
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sales_insight.dicionario_dados import DICIONARIO_DADOS

print("=== Ambiente de execução ===")
print(f"Python: {platform.python_version()}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy:  {np.__version__}")

=== Ambiente de execução ===
Python: 3.13.14
Pandas: 3.0.5
NumPy:  2.5.1


## 2. Carregamento e Inspeção dos Dados

### `RF01` — Carregamento do Dataset

O `vendas.csv` tem como base o [E-commerce Analytics Dataset Brazil](https://www.kaggle.com/datasets/joocarlosjr/e-commerce-analytics-dataset-brazil). Os dados foram consolidados, adaptados e enriquecidos para representar informações comerciais e logísticas.

Também foram introduzidas inconsistências controladas para viabilizar as etapas de limpeza e validação. O arquivo bruto permanece preservado em `data/raw`.


In [51]:
caminho_dados = Path("../data/raw/vendas.csv")

df_bruto = pd.read_csv(caminho_dados)
df = df_bruto.copy()

### `RF02` — Inspeção Estrutural

A inspeção inicial apresenta uma amostra dos registros, o dicionário de dados, as dimensões da base, os tipos das colunas e a ocorrência de valores ausentes.


In [52]:
display(df.head(3).style.hide(axis="index"))

id_venda,data_venda,id_cliente,nome_cliente,cidade,estado,regiao,id_produto,produto,categoria,quantidade,preco_unitario,desconto,previsao_entrega,data_entrega
ORD00001,2025-01-01,Cliente_008,Thales Pereira,Cascavel,PR,Sul,P0005,"Fone de Ouvido Sem Fio TWS, PHILIPS",Áudio,1.000000,124.910000,0.032700,2025-01-07,2025-01-09
ORD00002,2025-01-01,Cliente_011,Julia Ribeiro,Santa Maria,RS,Sul,P0006,Fone de ouvido Sem Fio QCY T27,Áudio,4.000000,129.860000,0.047700,2025-01-04,2025-01-08
ORD00003,2025-01-01,Cliente_019,Gabriela da Paz,Uberlândia,MG,Sudeste,P0002,Samsung Galaxy Tab S6 Lite,Smartphones e Tablets,2.000000,1799.010000,0.148100,2025-01-05,2025-01-04


#### Dicionário dos Dados


In [53]:
display(pd.DataFrame(DICIONARIO_DADOS).style.hide(axis="index"))

Coluna,Descrição
id_venda,Identificador único da venda.
data_venda,Data em que a venda foi realizada.
id_cliente,Identificador do cliente associado à venda.
nome_cliente,Nome do cliente.
cidade,Cidade de residência do cliente.
estado,Estado de residência do cliente.
regiao,Região geográfica associada ao cliente.
id_produto,Identificador do produto vendido.
produto,Nome do produto vendido.
categoria,Categoria à qual o produto pertence.


#### Dimensões, Tipos e Valores Ausentes


In [55]:
def resumo_estrutural(dataframe):
    """
    Retorna os tipos e valores ausentes das colunas.
    """
    return pd.DataFrame(
        {
            "coluna": dataframe.columns,
            "tipo": dataframe.dtypes.astype(str).values,
            "valores_ausentes": dataframe.isna().sum().values,
            "percentual_ausente": dataframe.isna().mean().values * 100,
        }
    )


linhas, colunas = df.shape
resumo_inicial = resumo_estrutural(df)

print(f"Dimensões: {linhas} linhas × {colunas} colunas")

display(
    resumo_inicial.style.hide(axis="index").format({"percentual_ausente": "{:.2f}%"})
)

Dimensões: 5732 linhas × 15 colunas


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,str,63,1.10%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


#### Diagnóstico Inicial

A base bruta contém 5.732 registros e 15 colunas. As colunas `data_venda`, `previsao_entrega` e `data_entrega` foram carregadas como texto e precisam ser convertidas para `datetime`.

Foram identificados valores ausentes em `data_venda` (63), `quantidade` (229) e `preco_unitario` (86). Esses pontos definem o escopo da limpeza realizada no RF03.


## 3. Limpeza e Tratamento dos Dados

### `RF03` — Padronização e Validação

A limpeza contempla a normalização de identificadores e textos, conversão das datas, validação das variáveis numéricas e remoção de registros incompletos nas colunas críticas.


In [56]:
registros_iniciais = len(df)

padroes_identificadores = {
    "id_venda": re.compile(r"^ORD\d{5}$"),
    "id_cliente": re.compile(r"^Cliente_\d{3}$"),
    "id_produto": re.compile(r"^P\d{4}$"),
}


def validar_identificadores(dataframe, padroes):
    """
    Retorna o resultado da validação dos identificadores.
    """
    resultados = []

    for coluna, padrao in padroes.items():
        validos = dataframe[coluna].astype("string").str.fullmatch(padrao, na=False)

        resultados.append(
            {
                "coluna": coluna,
                "valores_ausentes": int(dataframe[coluna].isna().sum()),
                "fora_do_padrao": int((~validos).sum()),
                "duplicados": (
                    int(dataframe[coluna].duplicated().sum())
                    if coluna == "id_venda"
                    else pd.NA
                ),
            }
        )

    return pd.DataFrame(resultados)


validacao_ids_antes = validar_identificadores(
    df,
    padroes_identificadores,
)

ids_clientes_corrigidos = int(
    validacao_ids_antes.loc[
        validacao_ids_antes["coluna"] == "id_cliente",
        "fora_do_padrao",
    ].iloc[0]
)

display(validacao_ids_antes.style.hide(axis="index"))

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,226,
id_produto,0,0,


#### 3.1 Padronização dos Identificadores e Textos

A validação identificou 226 ocorrências de `id_cliente` fora do padrão esperado. Esses valores serão normalizados com expressão regular. As demais colunas textuais terão espaços excedentes removidos.


In [57]:
def normalizar_id_cliente(valor):
    """Padroniza o identificador para o formato Cliente_000."""
    return re.sub(
        r"^cliente\D*(\d{3})\D*$",
        r"Cliente_\1",
        str(valor).strip(),
        flags=re.IGNORECASE,
    )


df["id_cliente"] = df["id_cliente"].apply(normalizar_id_cliente)

colunas_textuais = [
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "produto",
    "categoria",
]

resultado_padronizacao = []

for coluna in colunas_textuais:
    original = df[coluna].copy()
    padronizado = original.str.strip()

    resultado_padronizacao.append(
        {
            "coluna": coluna,
            "unicos_antes": original.nunique(),
            "unicos_depois": padronizado.nunique(),
            "espacos_corrigidos": int(original.ne(padronizado).sum()),
        }
    )

    df[coluna] = padronizado

resumo_padronizacao = pd.DataFrame(resultado_padronizacao)

validacao_ids_depois = validar_identificadores(
    df,
    padroes_identificadores,
)

display(validacao_ids_depois.style.hide(axis="index"))

display(resumo_padronizacao.style.hide(axis="index"))

coluna,valores_ausentes,fora_do_padrao,duplicados
id_venda,0,0,0
id_cliente,0,0,
id_produto,0,0,


coluna,unicos_antes,unicos_depois,espacos_corrigidos
nome_cliente,530,530,0
cidade,105,105,0
estado,22,22,0
regiao,4,4,0
produto,72,27,103
categoria,15,5,69


#### 3.2 Conversão das Datas

As colunas temporais serão convertidas para `datetime`. O parâmetro `errors="coerce"` transforma valores incompatíveis em `NaT`, permitindo identificar falhas de conversão.


In [58]:
colunas_datas = [
    "data_venda",
    "previsao_entrega",
    "data_entrega",
]

datas_originais = df[colunas_datas].copy()

df[colunas_datas] = df[colunas_datas].apply(
    pd.to_datetime,
    errors="coerce",
)

falhas_conversao = (df[colunas_datas].isna() & datas_originais.notna()).sum()

validacao_datas = pd.DataFrame(
    {
        "coluna": colunas_datas,
        "tipo_final": [str(df[coluna].dtype) for coluna in colunas_datas],
        "falhas_conversao": falhas_conversao.values,
    }
)

display(validacao_datas.style.hide(axis="index"))

coluna,tipo_final,falhas_conversao
data_venda,datetime64[us],0
previsao_entrega,datetime64[us],0
data_entrega,datetime64[us],0


#### 3.3 Tratamento das Ausências e Validação Numérica

Os registros com ausência em `data_venda`, `quantidade` ou `preco_unitario` serão removidos. Como existem linhas com ausência em mais de uma dessas colunas, a remoção considera os registros únicos afetados.


In [59]:
colunas_criticas = [
    "data_venda",
    "quantidade",
    "preco_unitario",
]

ausencias_criticas = (
    df[colunas_criticas]
    .isna()
    .sum()
    .rename_axis("coluna")
    .reset_index(name="valores_ausentes")
)

mascara_remocao = df[colunas_criticas].isna().any(axis=1)

total_removidos = int(mascara_remocao.sum())

df = df.loc[~mascara_remocao].copy()

display(ausencias_criticas.style.hide(axis="index"))

print(f"Registros únicos removidos: {total_removidos}")

coluna,valores_ausentes
data_venda,63
quantidade,229
preco_unitario,86


Registros únicos removidos: 371


In [60]:
validacoes_numericas = {
    "quantidade_inteira": bool((df["quantidade"] % 1 == 0).all()),
    "quantidade_positiva": bool((df["quantidade"] > 0).all()),
    "preco_positivo": bool((df["preco_unitario"] > 0).all()),
    "desconto_valido": bool(df["desconto"].between(0, 1).all()),
}

if not all(validacoes_numericas.values()):
    raise ValueError("Foram encontrados valores numéricos inválidos.")

df["quantidade"] = df["quantidade"].astype("int64")

colunas_numericas = [
    "quantidade",
    "preco_unitario",
    "desconto",
]

validacao_numerica = pd.DataFrame(
    {
        "coluna": colunas_numericas,
        "tipo": (df[colunas_numericas].dtypes.astype(str).values),
        "valor_minimo": [df[coluna].min() for coluna in colunas_numericas],
        "valor_maximo": [df[coluna].max() for coluna in colunas_numericas],
        "regra_validada": [
            (
                validacoes_numericas["quantidade_inteira"]
                and validacoes_numericas["quantidade_positiva"]
            ),
            validacoes_numericas["preco_positivo"],
            validacoes_numericas["desconto_valido"],
        ],
    }
)

display(validacao_numerica.style.hide(axis="index"))

coluna,tipo,valor_minimo,valor_maximo,regra_validada
quantidade,int64,1.000000,4.000000,True
preco_unitario,float64,17.900000,4604.000000,True
desconto,float64,0.000100,0.350000,True


#### Resultado da Limpeza


In [61]:
registros_finais = len(df)

espacos_corrigidos = {
    linha.coluna: linha.espacos_corrigidos for linha in resumo_padronizacao.itertuples()
}

relatorio_limpeza = {
    "registros_iniciais": registros_iniciais,
    "ids_cliente_normalizados": ids_clientes_corrigidos,
    "espacos_produto_corrigidos": espacos_corrigidos["produto"],
    "espacos_categoria_corrigidos": espacos_corrigidos["categoria"],
    "registros_removidos": total_removidos,
    "registros_finais": registros_finais,
}

tabela_relatorio_limpeza = pd.DataFrame(
    relatorio_limpeza.items(),
    columns=["indicador", "valor"],
)

display(tabela_relatorio_limpeza.style.hide(axis="index"))

display(
    resumo_estrutural(df)
    .style.hide(axis="index")
    .format({"percentual_ausente": "{:.2f}%"})
)

indicador,valor
registros_iniciais,5732
ids_cliente_normalizados,226
espacos_produto_corrigidos,103
espacos_categoria_corrigidos,69
registros_removidos,371
registros_finais,5361


coluna,tipo,valores_ausentes,percentual_ausente
id_venda,str,0,0.00%
data_venda,datetime64[us],0,0.00%
id_cliente,str,0,0.00%
nome_cliente,str,0,0.00%
cidade,str,0,0.00%
estado,str,0,0.00%
regiao,str,0,0.00%
id_produto,str,0,0.00%
produto,str,0,0.00%
categoria,str,0,0.00%


Foram normalizados 226 identificadores de clientes, 103 ocorrências com espaços excedentes em produtos e 69 em categorias. As datas foram convertidas sem falhas de formato.

A remoção das ausências críticas eliminou 371 registros únicos, mantendo 5.361 vendas válidas para as etapas seguintes.


## 3. Transformação e Criação de Colunas Derivadas

### `RF04` - Criar Colunas Derivadas com Transformações Condicionais

#### 1. **`receita_total` & `valor_desconto` & `faixa_receita_item`**

Como cada registro é considerado como uma venda efetuada e entregue, vou adicionar uma nova coluna `receita_total` = `quantidade` * (`preco_unitario` - (`preco_unitario` * `desconto`))

In [23]:
receita_total = df.quantidade * (
    df.preco_unitario - (df.preco_unitario * df.desconto)
).round(2)

df["receita_total"] = receita_total

valor_desconto = (df.preco_unitario * df.quantidade) - df.receita_total

df["valor_desconto"] = valor_desconto

Nesta etapa, a coluna `receita_total` será utilizada para criar uma classificação categórica do valor de cada venda.

A nova coluna `faixa_receita_item` dividirá as transações em três grupos:

- **Baixo Valor:** receita inferior a R\$ 500,00;
- **Médio Valor:** receita entre R\$ 500,00 e R\$ 4.999,99;
- **Alto Valor:** receita igual ou superior a R\$ 5.000,00.

A classificação será feita com `np.select`, permitindo aplicar múltiplas condições de forma vetorizada sobre o DataFrame, sem a necessidade de percorrer os registros com laços.


In [24]:
condicoes = [
    df.receita_total < 500,
    (df.receita_total >= 500) & (df.receita_total < 5000),
    df.receita_total >= 5000,
]

faixas = ["Baixo Valor", "Médio Valor", "Alto Valor"]

df["faixa_receita_item"] = np.select(condicoes, faixas, default="Não Classificado")

#### 2. **`mes_venda` & `bimestre_venda`**

- determinar intervalo temporal
- gerar coluna com nome do mês `mes_venda`

In [25]:
# Detreminar intervalo temporal
primeiro_mes = df.data_venda.min().month
ultimo_mes = df.data_venda.max().month

print(
    f"""
O intervalo de vendas é compreendido entre os mês {primeiro_mes} até mês {ultimo_mes}
"""
)


O intervalo de vendas é compreendido entre os mês 1 até mês 6



Como o dataset compreende um intervalo temporal que vai do mês 1 (Janeiro) até o mês 6 (Junho). Gerar coluna para `bimestre_venda`:

- `B1` Janeiro e Fevereiro
- `B2` Março e Abril
- `B3` Maio e Junho

In [26]:
meses = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril", 5: "Maio", 6: "Junho"}
bimestres = {1: "B1", 2: "B1", 3: "B2", 4: "B2", 5: "B3", 6: "B3"}

df["mes_venda"] = df["data_venda"].dt.month.map(meses)

df["bimestre"] = df["data_venda"].dt.month.map(bimestres)

#### 3. **`desvio_entrega_dias` & `atrasado`**

Nesta etapa, serão criadas duas colunas derivadas a partir de `previsao_entrega` e `data_entrega`.

A coluna `desvio_entrega_dias` representará a diferença, em dias, entre a entrega realizada e a previsão:

* valor negativo: entrega antecipada;
* valor zero: entrega no prazo;
* valor positivo: entrega atrasada.

Também será criada a coluna booleana `atrasado`, indicando diretamente se a entrega ocorreu após a data prevista.

Essas duas colunas permitem analisar tanto a intensidade do desvio quanto a ocorrência de atraso.


In [27]:
df["desvio_entrega_dias"] = (df.data_entrega - df.previsao_entrega).dt.days

df["atrasado"] = df.desvio_entrega_dias > 0

#### Organizar colunas e salvar `vendas_processado.csv`

**Organizar Colunas**

In [28]:
ordem_colunas = [
    "id_venda",
    "data_venda",
    "mes_venda",
    "bimestre",
    "id_cliente",
    "nome_cliente",
    "cidade",
    "estado",
    "regiao",
    "id_produto",
    "produto",
    "categoria",
    "quantidade",
    "preco_unitario",
    "receita_total",
    "faixa_receita_item",
    "desconto",
    "valor_desconto",
    "previsao_entrega",
    "data_entrega",
    "desvio_entrega_dias",
    "atrasado",
]

df = df[ordem_colunas]

In [29]:
print(f"{"="*6} Novas Dimensões {"="*6}")
linhas, colunas = df.shape
print(f"{"Linhas":<7}: {linhas}\n{"Colunas":<7}: {colunas}")
print("=" * 23)

====== Novas Dimensões ======
Linhas : 5361
Colunas: 22


Desde o dataset bruto até esta etapa de processamento, foram adicionadas **7 novas colunas derivadas**, construídas a partir das informações já existentes. Essas variáveis ampliam o conjunto original com indicadores temporais, financeiros, logísticos e categóricos, preparando o DataFrame para as próximas etapas de análise e agregação.


In [30]:
pasta_processed = Path("../data/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_processed / "vendas_processado.csv"

df.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"CSV salvo - Pasta `data/processed`")

CSV salvo - Pasta `data/processed`


## 4. Análise e Agregação das Métricas de Vendas

Nas etapas anteriores, os dados de vendas foram carregados, inspecionados, limpos e transformados. As inconsistências relevantes foram tratadas e novas variáveis foram criadas para representar informações financeiras, temporais, categóricas e logísticas.

Com isso, o DataFrame processado deixa de ser apenas um conjunto de registros individuais e passa a constituir uma base preparada para análise. A partir desta etapa, o objetivo é reunir as vendas em grupos capazes de responder a questões relacionadas ao desempenho comercial do período.

### `RF05` — Calcular Métricas Agregadas com `groupby`

Cada linha do DataFrame representa uma venda individual. Entretanto, a observação isolada dessas transações não permite identificar diretamente tendências ou comparar o desempenho entre diferentes dimensões do negócio.

Para transformar os registros em informações analíticas, serão utilizadas operações de agrupamento com o método `groupby()` do Pandas. As vendas serão consolidadas para responder às seguintes questões:

* Como a receita, a quantidade vendida e o número de vendas se distribuem entre os meses?
* Quais são os cinco produtos que mais geraram receita?
* Qual foi a receita total obtida por cada categoria?
* Como as regiões se comparam em receita total e ticket médio?

Cada agrupamento resultará em um novo DataFrame, organizado em formato tabular. Ao final, essas tabelas serão reunidas em um dicionário, formando uma estrutura reutilizável para apresentação dos resultados e para as próximas etapas de visualização e exportação.

In [31]:
# atribuir o novo DataFrame
df = pd.read_csv("../data/processed/vendas_processado.csv")
df.head(2)

,id_venda,data_venda,mes_venda,bimestre,id_cliente,nome_cliente,cidade,estado,regiao,id_produto,...,quantidade,preco_unitario,receita_total,faixa_receita_item,desconto,valor_desconto,previsao_entrega,data_entrega,desvio_entrega_dias,atrasado
0,ORD00001,2025-01-01,Janeiro,B1,Cliente_008,Thales Pereira,Cascavel,PR,Sul,P0005,...,1,124.91,120.83,Baixo Valor,0.0327,4.08,2025-01-07,2025-01-09,2,True
1,ORD00002,2025-01-01,Janeiro,B1,Cliente_011,Julia Ribeiro,Santa Maria,RS,Sul,P0006,...,4,129.86,494.68,Baixo Valor,0.0477,24.76,2025-01-04,2025-01-08,4,True


**Como a receita, a quantidade vendida e o número de vendas se distribuem entre os meses?**

In [32]:
ordem_meses = [
    "Janeiro",
    "Fevereiro",
    "Março",
    "Abril",
    "Maio",
    "Junho",
]

In [33]:
relatorio_mensal = (
    df.groupby("mes_venda")
    .agg(
        receita=("receita_total", "sum"),
        unidades_vendidas=("quantidade", "sum"),
        numero_vendas=("id_venda", "nunique"),
    )
    .reset_index()
)

relatorio_mensal["mes_venda"] = pd.Categorical(
    relatorio_mensal["mes_venda"], categories=ordem_meses, ordered=True
)

relatorio_mensal = relatorio_mensal.sort_values("mes_venda").reset_index(drop=True)

display(relatorio_mensal.style.hide(axis="index").format({"receita": "{:.2f}"}))

mes_venda,receita,unidades_vendidas,numero_vendas
Janeiro,1299266.31,1968,824
Fevereiro,1591691.83,2063,855
Março,2291715.46,2443,991
Abril,2004674.61,2241,900
Maio,1613016.68,2081,880
Junho,1537652.30,2131,911


**Quais são os cinco produtos que mais geraram receita?**

In [34]:
top_produto_receita = (
    df.groupby("produto")
    .agg(
        receita_total=("receita_total", "sum"),
    )
    .reset_index()
)

top_produto_receita = top_produto_receita.sort_values(
    "receita_total", ascending=False
).reset_index(drop=True)

top_5 = top_produto_receita.head(5)

for posicao, linha in enumerate(
    top_5.itertuples(index=False),
    start=1,
):
    print(
        f"{posicao:2}º {linha.produto:<40} — " f"Receita: R$ {linha.receita_total:,.2f}"
    )

 1º ACER Notebook Gamer Nitro                — Receita: R$ 1,848,972.94
 2º Projetor Smart Epson EpiqVision, FULL HD — Receita: R$ 1,244,636.36
 3º Samsung Galaxy A36                       — Receita: R$ 850,244.53
 4º Samsung Galaxy Tab S6 Lite               — Receita: R$ 839,934.77
 5º Projetor EPSON Powerlite Wide Screen     — Receita: R$ 801,184.04


**Qual foi a receita total obtida por cada categoria?**

In [35]:
receita_por_categoria = df.groupby("categoria").agg(
    receita_total=("receita_total", "sum")
)

display(receita_por_categoria.sort_values("receita_total", ascending=False))

,receita_total
categoria,
Informática,3049354.76
TV e Projeção,3022977.33
Áudio,2140350.33
Smartphones e Tablets,2017909.85
Acessórios Mobile,107424.92


**Como as regiões se comparam em receita total e ticket médio?**

In [36]:
relatorio_regional = df.groupby("regiao", as_index=False).agg(
    receita_total=("receita_total", "sum"),
    numero_vendas=("id_venda", "nunique"),
)

relatorio_regional["ticket_medio"] = (
    relatorio_regional.receita_total / relatorio_regional.numero_vendas
).round(2)

relatorio_regional.sort_values("ticket_medio", ascending=False)

,regiao,receita_total,numero_vendas,ticket_medio
2,Sudeste,2829006.77,1435,1971.43
1,Norte,2393107.36,1232,1942.46
0,Nordeste,2339772.12,1217,1922.57
3,Sul,2776130.94,1477,1879.57


#### Métricas Gerais

In [37]:
metricas_gerais = {
    "relatorio_mensal": relatorio_mensal,
    "top_produtos": top_produto_receita,
    "receita_categoria": receita_por_categoria,
    "relatorio_regional": relatorio_regional,
}

### `RF06` — Segmentar Clientes por Nível de Gasto

Para identificar os clientes de maior valor, as vendas serão agrupadas por cliente e a receita de suas transações será acumulada. Em seguida, cada cliente será classificado de acordo com seu gasto total.

| Gasto total acumulado         | Segmento |
| ----------------------------- | -------- |
| Abaixo de R\$ 5.000,00         | Bronze   |
| De R\$ 5.000,00 a R\$ 15.000,00 | Prata    |
| Acima de R\$ 15.000,00         | Ouro     |

A classificação será aplicada com uma função `lambda`, resultando em uma estrutura com o identificador, o nome, o gasto total e o segmento de cada cliente.


In [38]:
segmentacao_cliente = df.groupby(["id_cliente", "nome_cliente"], as_index=False).agg(
    gasto_total=("receita_total", "sum")
)

segmentacao_cliente["segmento"] = segmentacao_cliente.gasto_total.apply(
    lambda gasto: (
        "Bronze" if gasto < 5_000 else "Prata" if gasto <= 15_000 else "Ouro"
    )
)

segmentacao_cliente = segmentacao_cliente.sort_values(
    "gasto_total", ascending=False
).reset_index(drop=True)

#### Clientes com maior gasto acumulado

A segmentação completa será ordenada pelo gasto total em ordem decrescente. A tabela seguinte apresenta os dez clientes que mais geraram receita no período analisado.


In [39]:
top_10_clientes = segmentacao_cliente.head(10)

display(top_10_clientes.style.hide(axis="index").format({"gasto_total": "R$ {:,.2f}"}))

id_cliente,nome_cliente,gasto_total,segmento
Cliente_371,Ana Luiza Nunes,"R$ 116,525.85",Ouro
Cliente_165,Maria Fernanda Teixeira,"R$ 111,976.65",Ouro
Cliente_394,Eduardo Oliveira,"R$ 108,436.37",Ouro
Cliente_008,Thales Pereira,"R$ 96,223.86",Ouro
Cliente_360,Mariane Campos,"R$ 92,852.62",Ouro
Cliente_300,Dra. Maria Sophia Nogueira,"R$ 89,455.87",Ouro
Cliente_218,Isabel Pereira,"R$ 89,303.07",Ouro
Cliente_337,Benício Fernandes,"R$ 87,821.42",Ouro
Cliente_266,Caroline Carvalho,"R$ 87,502.24",Ouro
Cliente_357,Amanda Araújo,"R$ 81,303.37",Ouro


#### Distribuição dos clientes por segmento

Além do ranking individual, a contagem de clientes em cada segmento permite observar como a base está distribuída entre as três faixas de gasto.


In [40]:
distribuicao_segmentos = (
    segmentacao_cliente["segmento"]
    .value_counts()
    .rename_axis("segmento")
    .reset_index(name="numero_clientes")
)

display(distribuicao_segmentos.style.hide(axis="index"))

segmento,numero_clientes
Ouro,245
Prata,182
Bronze,107


#### Síntese da segmentação

Foram classificados 534 clientes: 245 no segmento Ouro, 182 no segmento Prata e 107 no segmento Bronze. O segmento Ouro representa a maior parcela da base, com aproximadamente 45,9% dos clientes.

Ana Luiza Nunes apresentou o maior gasto acumulado, com R$ 116.525,85. Todos os clientes do Top 10 pertencem ao segmento Ouro, evidenciando a concentração dos maiores valores na faixa superior da classificação.

## 5. Operações Numéricas com NumPy

### `RF07` — Aplicar Operações Vetorizadas e Broadcasting

Nesta etapa, a coluna `receita_total` será convertida de uma `Series` do Pandas para um array NumPy. Essa estrutura permitirá aplicar operações numéricas diretamente sobre todos os valores, sem percorrer individualmente cada elemento.

O array será utilizado para calcular estatísticas gerais, realizar uma transformação vetorizada com broadcasting e filtrar as vendas que apresentam receita acima da média.


In [41]:
receitas_array = df.receita_total.to_numpy()

In [42]:
# Tipo dos dados
receitas_array.dtype

dtype('float64')

In [43]:
receitas_array.ndim

1

In [44]:
receitas_array.size

5361

#### Estatísticas gerais da receita

A média, a mediana e o desvio-padrão serão calculados diretamente sobre o array de receitas. Os resultados serão armazenados em um dicionário reutilizável e apresentados em formato tabular.


In [45]:
estatisticas_receita = {
    "media": float(np.mean(receitas_array)),
    "mediana": float(np.median(receitas_array)),
    "desvio": float(np.std(receitas_array)),
}

tabela_estatisticas = (
    pd.DataFrame.from_dict(estatisticas_receita, orient="index", columns=["valor"])
    .rename_axis("medida")
    .reset_index()
)

display(tabela_estatisticas.style.hide(axis="index").format({"valor": "R$ {:,.2f}"}))

medida,valor
media,"R$ 1,928.37"
mediana,R$ 979.70
desvio,"R$ 2,530.21"


A receita média por venda foi de R$ 1.928,37, enquanto a mediana foi de R$ 979,70. A diferença entre essas medidas indica que vendas de maior valor elevam a média, enquanto pelo menos metade das transações permanece abaixo de aproximadamente R$ 980,00.

O desvio-padrão de R$ 2.530,21 demonstra uma dispersão elevada entre os valores das vendas.

#### Normalização vetorizada das receitas

As receitas serão normalizadas pelo método Min–Max, que transforma os valores originais para o intervalo entre 0 e 1. A operação será aplicada diretamente sobre o array, sem laços, utilizando o valor mínimo e o valor máximo como escalares.

A aplicação desses escalares a todos os elementos demonstra o broadcasting do NumPy, enquanto o cálculo simultâneo caracteriza a operação vetorizada.


$$x_{\text{normalizado}} = \dfrac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$

In [46]:
valor_min = receitas_array.min()
valor_max = receitas_array.max()

receitas_normalizads = (receitas_array - valor_min) / (valor_max - valor_min)

print(
    f"Máximo deve ser 1.0: {receitas_normalizads.max()}\nMínimo deve ser 0.0: {receitas_normalizads.min()}"
)

Máximo deve ser 1.0: 1.0
Mínimo deve ser 0.0: 0.0


#### Vendas com receita acima da média

A média calculada anteriormente será utilizada como referência para identificar as vendas de maior valor. A comparação entre o array de receitas e esse valor produzirá uma máscara booleana, utilizada para selecionar e contar somente as transações com receita acima da média.


In [47]:
valor_medio = estatisticas_receita["media"]

mask_booleana = receitas_array > valor_medio

vendas_acima = receitas_array[mask_booleana]

numero_vendas = len(receitas_array)
numero_vendas_acima = len(vendas_acima)
pct_vendas_acima = round((numero_vendas_acima / numero_vendas) * 100, 2)

print(
    f"""
==================   Resultados   ===================
Número de vendas total:                  {numero_vendas}
Número de vendas receita acima da média: {numero_vendas_acima}
Porcentagem de vendas acima da média   : {pct_vendas_acima}%
"""
)


==================   Resultados   ===================
Número de vendas total:                  5361
Número de vendas receita acima da média: 1720
Porcentagem de vendas acima da média   : 32.08%



#### Síntese das operações NumPy

O array analisado contém 5.361 valores do tipo `float64`. A normalização Min–Max transformou as receitas para o intervalo entre 0 e 1, confirmando os limites mínimo e máximo esperados.

A filtragem booleana identificou 1.720 vendas com receita superior à média de R$ 1.928,37, correspondendo a 32,08% das transações. Todo o processamento foi realizado diretamente sobre o array, sem laços, demonstrando agregações, vetorização, broadcasting e seleção por máscara booleana.

## 6. Visualização dos Resultados

### `RF08` — Criar Visualizações com Matplotlib e Seaborn

As métricas calculadas anteriormente são apresentadas graficamente para destacar tendências, diferenças de desempenho e relações entre variáveis.

#### Evolução da Receita Mensal

Como a receita se comportou ao longo do período analisado?

![Receita total por mês](../reports/figures/receita_por_mes.png)

A receita cresceu de janeiro até março, quando atingiu aproximadamente R$ 2,29 milhões. Após o pico, houve redução gradual entre abril e junho.

#### Produtos com Maior Receita

Quais produtos tiveram maior participação na receita total?

![Produtos com maior receita](../reports/figures/top_produtos.png)

O notebook gamer Acer Nitro apresentou a maior receita, com aproximadamente R$ 1,85 milhão, seguido pelo projetor Epson EpiqVision. Os resultados mostram forte concentração da receita em produtos de maior valor unitário.

#### Relação entre Quantidade e Receita

Vendas com maior quantidade geram necessariamente maior receita?

![Relação entre quantidade e receita](../reports/figures/quantidade_vs_receita.png)

A relação positiva entre quantidade e receita é fraca. O valor da transação também depende do preço dos produtos, permitindo receitas elevadas mesmo em vendas com quantidades moderadas.

#### Cidades com Maior Desempenho Comercial

Quais cidades concentram o maior número de vendas e as maiores receitas?

![Cidades com mais vendas e receita](../reports/figures/top_cidades_vendas_receita.png)

Ribeirão Preto liderou os dois rankings, seguida por Cascavel. As diferenças nas demais posições mostram que um maior número de vendas nem sempre resulta em maior receita, pois o valor das transações também influencia o desempenho de cada cidade.
